# python-bazi 검증

음력 변환·사주 계산·분석 결과 확인

In [1]:
import sys
sys.path.insert(0, '..')
import bazi
from datetime import datetime, date
print('bazi 버전:', bazi.__version__)

bazi 버전: 0.1.0


## 1. 사주 계산 기본

In [2]:
samples = [
    ('1900-01-31 00:00', datetime(1900, 1, 31, 0, 0)),
    ('1992-08-04 03:30 (hjseo)', datetime(1992, 8, 4, 3, 30)),
    ('2000-01-01 00:00', datetime(2000, 1, 1, 0, 0)),
    ('2024-02-10 12:00 (설날)', datetime(2024, 2, 10, 12, 0)),
]
for label, dt in samples:
    c = bazi.chart(dt)
    print(f'{label}: {c}')

1900-01-31 00:00: 己亥 丁丑 甲辰 甲子
1992-08-04 03:30 (hjseo): 壬申 丁未 壬子 壬寅
2000-01-01 00:00: 己卯 丙子 戊午 壬子
2024-02-10 12:00 (설날): 甲辰 丙寅 甲辰 庚午


## 2. 음력 입력 = 표준시 입력 일치 확인

In [3]:
checks = [
    (datetime(1992, 8, 4, 3, 30), datetime(1992, 7, 6, 3, 30)),
    (datetime(2024, 2, 10, 12, 0), datetime(2024, 1, 1, 12, 0)),
]
for solar_dt, lunar_dt in checks:
    c1 = bazi.chart(solar_dt)
    c2 = bazi.chart(lunar_dt, time_basis='lunar')
    ok = '✓' if c1 == c2 else '✗'
    print(f'{ok} 양력 {solar_dt.date()} = 음력 {lunar_dt.date()}: {c1}')

✓ 양력 1992-08-04 = 음력 1992-07-06: 壬申 丁未 壬子 壬寅
✓ 양력 2024-02-10 = 음력 2024-01-01: 甲辰 丙寅 甲辰 庚午


## 3. 음력 변환 — 주요 날짜

In [4]:
import pandas as pd

samples = [
    (1900, 1, 1, False),
    (1992, 7, 6, False),
    (2000, 1, 1, False),
    (2020, 3, 1, False),
    (2020, 4, 1, True),   # 윤4월
    (2020, 4, 1, False),  # 4월
    (2023, 2, 1, True),   # 2023 윤2월
    (2099, 6, 15, False),
]

rows = []
for ly, lm, ld, leap in samples:
    label = f'{ly}-{"윤" if leap else ""}{lm}-{ld}'
    solar = bazi.lunar_to_solar(ly, lm, ld, is_leap=leap)
    back = bazi.solar_to_lunar(solar)
    rt_ok = back == (ly, lm, ld, leap)
    rows.append({'음력': label, '양력': str(solar), 'roundtrip': '✓' if rt_ok else f'✗ {back}'})

pd.DataFrame(rows)

,음력,양력,roundtrip
0,1900-1-1,1900-02-01,✓
1,1992-7-6,1992-08-04,✓
2,2000-1-1,2000-02-05,✓
3,2020-3-1,2020-03-24,✓
4,2020-윤4-1,2020-05-23,✓
5,2020-4-1,2020-04-23,✓
6,2023-윤2-1,2023-03-22,✓
7,2099-6-15,2099-08-01,✓


## 4. 전체 달력 roundtrip (1800 ~ 2199년)

In [5]:
import numpy as np

data = np.load('../bazi/_data/lunar_table_1800_2200.npz')
ordinals = data['ordinals']
years    = data['years']
months   = data['months']

failures = []
for i in range(len(ordinals)):
    solar = date.fromordinal(int(ordinals[i]))
    ly = int(years[i])
    mo = int(months[i])
    leap = mo < 0
    lm = abs(mo)
    back = bazi.solar_to_lunar(solar)
    if back != (ly, lm, 1, leap):
        failures.append({'row': i, 'expected': (ly, lm, 1, leap), 'got': back, 'solar': str(solar)})

print(f'총 {len(ordinals)}개 달 테스트')
if failures:
    print(f'실패: {len(failures)}개')
    for f in failures[:10]:
        print(f)
else:
    print('전체 통과 ✓')

총 4962개 달 테스트
전체 통과 ✓


## 5. 분석 — hjseo 사주

In [6]:
bazi.config.lang = 'ko'
c = bazi.chart(datetime(1992, 8, 4, 3, 30))
r = bazi.analyze(c, sex='male', birth=date(1992, 8, 4))

print('사주:', c)
print()

print('[오행]')
for elem, cnt in r.elements.counts.items():
    bar = '■' * cnt
    print(f'  {elem} {bar} ({cnt})')
print()

print('[기둥 분석]')
for key, p in r.pillars.items():
    print(f'  {key:5s}: {p.stem}{p.branch}  천간={p.stem_shi_shen}  지지={p.branch_shi_shen}  지장간={p.hidden_stems}')
print()

print('[대운]')
for dy in r.dayun:
    print(f'  {dy.start_age:2d}세: {dy.stem}{dy.branch}  ({dy.stem_shi_shen}/{dy.branch_shi_shen})')

사주: 壬申 丁未 壬子 壬寅

[오행]
  木 ■ (1)
  火 ■ (1)
  土 ■ (1)
  金 ■ (1)
  水 ■■■■ (4)

[기둥 분석]
  year : 壬申  천간=비견  지지=편인  지장간=['戊', '壬', '庚']
  month: 丁未  천간=정재  지지=정관  지장간=['丁', '乙', '己']
  day  : 壬子  천간=일원  지지=겁재  지장간=['壬', '癸']
  hour : 壬寅  천간=비견  지지=식신  지장간=['戊', '丙', '甲']

[대운]
   1세: 戊申  (편관/편인)
  11세: 己酉  (정관/정인)
  21세: 庚戌  (편인/편관)
  31세: 辛亥  (정인/비견)
  41세: 壬子  (비견/겁재)
  51세: 癸丑  (겁재/정인)
  61세: 甲寅  (식신/식신)
  71세: 乙卯  (상관/상관)


## 6. 시주 미상 — date 입력 (삼주)

In [7]:
from datetime import date

# date 타입 입력 → 시주 None
c_date = bazi.chart(date(1992, 8, 4))
print('삼주:', c_date)
print('hour:', c_date.hour)
print()

# analyze — 시주 없으면 3기둥만
bazi.config.lang = 'ko'
r = bazi.analyze(c_date, sex='male', birth=date(1992, 8, 4))

print('[오행] (3기둥 기준)')
for elem, cnt in r.elements.counts.items():
    if cnt:
        bar = '■' * cnt
        print(f'  {elem} {bar} ({cnt})')
print(f'  합계: {sum(r.elements.counts.values())}칸 (3기둥 × 천간+지지)')
print()

print('[기둥 분석]')
for key, p in r.pillars.items():
    print(f'  {key:5s}: {p.stem}{p.branch}  천간={p.stem_shi_shen}  지지={p.branch_shi_shen}')
print()

# datetime vs date 비교 (년·월·일 일치 확인)
c_dt = bazi.chart(datetime(1992, 8, 4, 3, 30))
ok = (c_date.year == c_dt.year and c_date.month == c_dt.month and c_date.day == c_dt.day)
print('년·월·일주 일치:', '✓' if ok else '✗')

삼주: 壬申 丁未 壬子
hour: None

[오행] (3기둥 기준)
  火 ■ (1)
  土 ■ (1)
  金 ■ (1)
  水 ■■■ (3)
  합계: 6칸 (3기둥 × 천간+지지)

[기둥 분석]
  year : 壬申  천간=비견  지지=편인
  month: 丁未  천간=정재  지지=정관
  day  : 壬子  천간=일원  지지=겁재

년·월·일주 일치: ✓
